In [95]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [96]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': False,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8]},
 'method': 'welch',
 'stepSize': 1.8,
 'windowLength': 2}

In [97]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.8, 'windowLength': 2}


In [98]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [99]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

Stopping existing Spark context...
Previous Spark context stopped successfully


In [100]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

New Spark session created successfully


25/04/09 13:07:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/09 13:07:50 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/04/09 13:07:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [101]:
# adding dependencies to spark's context so spark workers (the threads) can access them

In [102]:
from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [103]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participants

In [104]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Config not found in feature_extraction.py                         (0 + 12) / 12]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.8, 'windowLength': 2}
Processing subject sub-014
processSub sub-014
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.8, 'windowLength': 2}
Processing subject sub-022
processSub sub-022
Config not found in feature_extracti

Processed 1549640 records for Alzheimer's group
Processed 1287725 records for Control group
CPU times: user 192 ms, sys: 108 ms, total: 300 ms
Wall time: 2min 10s


Total rows collected: 48070
Returning DataFrame with 48070 rows
Column names from schema: ['SubjectID', 'EpochID', 'WaveBand', 'Electrode', 'Power']
4.9602391719818115
                                                                                

In [105]:
#just renaming things now that we understand the types and where things are coming from
alz_df = group_a_spark_df
cntrl_df = group_c_spark_df

In [106]:
alz_df.head()

Row(SubjectID='sub-008', EpochID='ep-0', WaveBand='Alpha', Electrode='Fp1', Power=0.0013661661162441853)

In [107]:
cntrl_df.head()

Row(SubjectID='sub-037', EpochID='ep-0', WaveBand='Alpha', Electrode='Fp1', Power=0.010359228054253546)

In [108]:
NUM_TEST_SUBJECTS_PER_GROUP = 2  # i know before we had three , but 2 is better 3 took out too much data.

alz_test_subjects = (
    alz_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .repartition(16)
    .collect()
)


cntrl_test_subjects = (
    cntrl_df.select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .repartition(16)
    .collect()
)


In [109]:
print(f"azl test subjects {alz_test_subjects}\ncntrl test subjects {cntrl_test_subjects}")

azl test subjects ['sub-001', 'sub-002']
cntrl test subjects ['sub-037', 'sub-038']


In [110]:
from pyspark.sql.functions import lit

In [111]:
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [112]:
# Filter test rows
alz_test_df = alz_df.filter(alz_df.SubjectID.isin(alz_test_subjects))
cntrl_test_df = cntrl_df.filter(cntrl_df.SubjectID.isin(cntrl_test_subjects))

# Filter training rows (not in test subjects)
alz_train_df = alz_df.filter(~alz_df.SubjectID.isin(alz_test_subjects))
cntrl_train_df = cntrl_df.filter(~cntrl_df.SubjectID.isin(cntrl_test_subjects))

In [113]:
# Individual counts
alz_train_count = alz_train_df.select("SubjectID", "EpochID").distinct().count()
alz_test_count = alz_test_df.select("SubjectID", "EpochID").distinct().count()
cntrl_train_count = cntrl_train_df.select("SubjectID", "EpochID").distinct().count()
cntrl_test_count = cntrl_test_df.select("SubjectID", "EpochID").distinct().count()

# Combined counts
test_total_count = alz_test_df.unionByName(cntrl_test_df).select("SubjectID", "EpochID").distinct().count()
train_total_count = alz_train_df.unionByName(cntrl_train_df).select("SubjectID", "EpochID").distinct().count()

# Print them out
print(f"Alzheimer's train: {alz_train_count}")
print(f"Alzheimer's test:  {alz_test_count}")
print(f"Control train:     {cntrl_train_count}")
print(f"Control test:      {cntrl_test_count}")
print(f"Total test:        {test_total_count}")
print(f"Total train:       {train_total_count}")

Alzheimer's train: 15539
Alzheimer's test:  773
Control train:     12628
Control test:      927
Total test:        1700
Total train:       28167


In [114]:
train_df = alz_train_df.unionByName(cntrl_train_df)
test_df = alz_test_df.unionByName(cntrl_test_df)

In [115]:
from dimensionality_reduction import normalize_power # it z-scores the data
# NOTE, this uses the first parameters for mean and std for the z-score, so none of test_df's data is used to z-score
train_df, test_df = normalize_power(train_df, test_df) 

In [116]:
from dimensionality_reduction import prepare_features_for_pca
# this pivots the tables so that its better suited for PCA and ML with pyspark's libraries
train_df, train_features_column = prepare_features_for_pca(train_df)
test_df, test_features_column = prepare_features_for_pca(test_df)

In [117]:
train_features_column.sort()
test_features_column.sort()
if train_features_column != test_features_column:
    print("!! VERY UNUSUAL, NEED TO DEBUG, it means that the trainig and testing have different columns :( !!")

In [118]:
from dimensionality_reduction import fit_pca_model
#Note , this finds the features to explain the model's PCA
K_VAR_TARGET=0.95
pca_model_func, k_val = fit_pca_model(train_df, train_features_column, variance_target=K_VAR_TARGET)

In [119]:
print(f"We can explian {K_VAR_TARGET} with {k_val} features. That is a lot less then {len(train_df.columns)-3} features we originally had (we hope).") #-3 for subjectID , epochID and lebel

We can explian 0.95 with 20 features. That is a lot less then 95 features we originally had (we hope).


In [120]:
# know that we know we can explain 95% of the variance (or what we set target to) ,
# lets make our dataframces only have those important columns
from dimensionality_reduction import apply_pca_model
train_df = apply_pca_model(train_df, train_features_column, pca_model_func, k_val)
test_df  = apply_pca_model(test_df, train_features_column, pca_model_func, k_val)

In [121]:
train_df.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = false)



# ML

In [122]:
model_summaries = []

In [123]:
%%time
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Get input size from PCA features

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=[k_val, 100, 2],  # input → hidden (100 units) → 2 output classes
    maxIter=1000,
    seed=42
)

mlp_model = mlp.fit(train_df)
mlp_preds = mlp_model.transform(test_df)

CPU times: user 13.2 ms, sys: 9.56 ms, total: 22.8 ms
Wall time: 1min 5s


In [124]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mlp_auc = evaluator.evaluate(mlp_preds)

print("MLP AUC:", mlp_auc)

MLP AUC: 0.6974326340306822


In [125]:
preds_pd = mlp_preds.select("prediction", "label").toPandas()

from sklearn.metrics import classification_report, accuracy_score

print("Neural Network accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

Neural Network accuracy: 0.6611764705882353
              precision    recall  f1-score   support

     Control       0.73      0.60      0.66       927
 Alzheimer's       0.61      0.73      0.66       773

    accuracy                           0.66      1700
   macro avg       0.67      0.67      0.66      1700
weighted avg       0.67      0.66      0.66      1700



In [126]:
from sklearn.metrics import classification_report

report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

model_summaries.append({
    "model": "Neural Network",
    "auc": mlp_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [127]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=100)
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
gbt_auc = evaluator.evaluate(gbt_preds)
print("Gradient Boosted Trees AUC:", gbt_auc)

preds_pd = gbt_preds.select("prediction", "label").toPandas()
print("GBT accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1000.2 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1002.9 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1003.3 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1004.0 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1005.0 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1007.3 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1010.0 KiB
25/04/09 13:11:35 WARN DAGScheduler: Broadcasting large task binary with size 1010.5 KiB
25/04/09 13:11:36 WARN DAGScheduler: Broadcasting large task binary with size 1011.2 KiB
25/04/09 13:11:36 WARN DAGScheduler: Broadcasting large task binary with size 1012.1 KiB
25/04/09 13:11:36 WARN DAGScheduler: Broadcasting large task binary with size 1013.9 KiB
25/04/09 13:11:36 WAR

Gradient Boosted Trees AUC: 0.7031292084106109
GBT accuracy: 0.6347058823529412
              precision    recall  f1-score   support

     Control       0.72      0.54      0.62       927
 Alzheimer's       0.58      0.75      0.65       773

    accuracy                           0.63      1700
   macro avg       0.65      0.64      0.63      1700
weighted avg       0.65      0.63      0.63      1700



25/04/09 13:11:39 WARN DAGScheduler: Broadcasting large task binary with size 1010.7 KiB


In [128]:
# Generate classification report for GBT
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append GBT results to summary
model_summaries.append({
    "model": "Gradient Boosted Trees",
    "auc": gbt_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [129]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)
tree_model = tree.fit(train_df)
tree_preds = tree_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
tree_auc = evaluator.evaluate(tree_preds)
print("Decision Tree AUC:", tree_auc)

preds_pd = tree_preds.select("prediction", "label").toPandas()
print("Decision Tree accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


Decision Tree AUC: 0.6196043378813822
Decision Tree accuracy: 0.6129411764705882
              precision    recall  f1-score   support

     Control       0.70      0.50      0.59       927
 Alzheimer's       0.56      0.74      0.64       773

    accuracy                           0.61      1700
   macro avg       0.63      0.62      0.61      1700
weighted avg       0.64      0.61      0.61      1700



In [130]:
# Generate classification report for Decision Tree
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append Decision Tree results to summary
model_summaries.append({
    "model": "Decision Tree",
    "auc": tree_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [131]:
from pyspark.ml.classification import LinearSVC

# Train SVM model
svm = LinearSVC(featuresCol="features", labelCol="label", maxIter=100, regParam=0.1)
svm_model = svm.fit(train_df)
svm_preds = svm_model.transform(test_df)

# Evaluate SVM
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
svm_auc = evaluator.evaluate(svm_preds)
print("SVM AUC:", svm_auc)

# Accuracy and report
preds_pd = svm_preds.select("prediction", "label").toPandas()
from sklearn.metrics import classification_report, accuracy_score

print("SVM accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


SVM AUC: 0.6935767704805245
SVM accuracy: 0.548235294117647
              precision    recall  f1-score   support

     Control       0.71      0.29      0.41       927
 Alzheimer's       0.50      0.86      0.63       773

    accuracy                           0.55      1700
   macro avg       0.61      0.57      0.52      1700
weighted avg       0.62      0.55      0.51      1700



In [132]:
# Generate classification report for SVM
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append SVM results to summary
model_summaries.append({
    "model": "SVM",
    "auc": svm_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [133]:
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import col

# Step 1: Fit LSH model on training data
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=0.25,
    numHashTables=6
)
lsh_model = lsh.fit(train_df)

# Step 2: Perform approximate similarity join between test and train
# This will find the approximate nearest neighbors of test samples in train set
similarities = lsh_model.approxSimilarityJoin(
    datasetA=test_df,
    datasetB=train_df,
    threshold=float("inf"),  # You can limit this if needed
    distCol="euclidean_distance"
)

# Step 3: For each test point, pick nearest neighbor (smallest distance)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("datasetA").orderBy("euclidean_distance")

nearest_neighbors = similarities \
    .withColumn("rank", row_number().over(window)) \
    .filter(col("rank") == 1)

# Step 4: Collect prediction from nearest training label
predictions = nearest_neighbors.select(
    col("datasetA.label").alias("true_label"),
    col("datasetB.label").alias("predicted_label")
)

# Step 5: Evaluate
preds_pd = predictions.toPandas()

from sklearn.metrics import accuracy_score, classification_report

print("Approximate KNN accuracy:", accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names))


[Stage 14396:============================================>        (10 + 2) / 12]

Approximate KNN accuracy: 0.6123529411764705
              precision    recall  f1-score   support

     Control       0.65      0.62      0.63       927
 Alzheimer's       0.57      0.61      0.59       773

    accuracy                           0.61      1700
   macro avg       0.61      0.61      0.61      1700
weighted avg       0.61      0.61      0.61      1700



In [134]:
# Ensure columns are correctly named
if "true_label" not in preds_pd.columns:
    preds_pd.columns = ["true_label", "predicted_label"]

# Generate classification report
report_str = classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names, output_dict=True)

# Append Approximate KNN results to summary
model_summaries.append({
    "model": "Approximate KNN",
    "auc": "N/A",
    "accuracy": accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [135]:
print("\n\n=== MODEL PERFORMANCE SUMMARY ===")
for summary in model_summaries:
    print(f"\nModel: {summary['model']}")
    
    auc = summary.get('auc', 'N/A')
    if isinstance(auc, (int, float)):
        print(f"  AUC:      {auc:.4f}")
    else:
        print(f"  AUC:      {auc}")

    acc = summary.get('accuracy', 'N/A')
    if isinstance(acc, (int, float)):
        print(f"  Accuracy: {acc:.4f}")
    else:
        print(f"  Accuracy: {acc}")
    
    print("\n  Classification Report:")
    print(summary.get("report_str", "  No report available"))

print("\n\n=== ACTIVE CONFIGURATION ===")
from pprint import pprint
pprint(load_config())




=== MODEL PERFORMANCE SUMMARY ===

Model: Neural Network
  AUC:      0.6974
  Accuracy: 0.6612

  Classification Report:
              precision    recall  f1-score   support

     Control       0.73      0.60      0.66       927
 Alzheimer's       0.61      0.73      0.66       773

    accuracy                           0.66      1700
   macro avg       0.67      0.67      0.66      1700
weighted avg       0.67      0.66      0.66      1700


Model: Gradient Boosted Trees
  AUC:      0.7031
  Accuracy: 0.6347

  Classification Report:
              precision    recall  f1-score   support

     Control       0.72      0.54      0.62       927
 Alzheimer's       0.58      0.75      0.65       773

    accuracy                           0.63      1700
   macro avg       0.65      0.64      0.63      1700
weighted avg       0.65      0.63      0.63      1700


Model: Decision Tree
  AUC:      0.6196
  Accuracy: 0.6129

  Classification Report:
              precision    recall  f1-scor